In [1]:
import coiled
import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging 
import numpy as np
import pytz
import dask
import re
import requests
import sparse
import time
import warnings
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy
from flox.xarray import xarray_reduce

import logging
import pygwalker as pyg

# T0 INSTALL FLOX:
# 1. In home directory (cd ~), downloaded flox tar.gz (because can't install the latest version using conda-forge for some reason): wget https://files.pythonhosted.org/packages/6e/34/6eea00e3f1de745c8adad5a3dafd46c3481294cff8699c20a9b8d80502ed/flox-0.10.4.tar.gz 
# 2. Installed using pip, but it's still putting it in the active Conda environment: pip install /home/dagibbs22/flox-0.10.4.tar.gz

# TO CREATE A NOTEBOOK IN A COILED CLUSTER
# coiled notebook start --region=us-east-1

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
# Zarr creation cluster
cluster = coiled.Cluster(
    name="vegetation_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=50,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                                 ╷                                            │
│   Package                       │ Note                                       │
│ ╶───────────────────────────────┼──────────────────────────────────────────╴ │
│   coiled_local_zonal_statistics │ Source wheel built from                    │
│                                 │ /mnt/c/GIS/git/AFOLU_GHG_flux_model/src/   │
│                                 │ LULUCF/scripts/zonal_statistics            │
│   flox                          │ Wheel built from ~/flox-0.10.4.tar.gz      │
│                                 ╵                                            │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ duckdb==0.9.2, but you have duckdb 1.3.0.      │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ gw-dsl-parser==0.1.8a0, but you have           │           │
│                 │ gw-dsl-parser 0.1.49.1.                        │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.3.3.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.12 linux-aarch64 on conda-forge   │           │
│   lz4           │ lz4~=4.4.5 has no install candidate for Python │ Warning   │
│                 │ 3.12 linux-aarch64 on conda-forge              │           │
│   dtale         │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ dtale 3.17.0 has requirement dash<=2.18.2;     │           │
│                 │ python_version > "3.7", but you have dash      │           │
│                 │ 3.0.4.                                         │           │
│                 │ dtale 3.17.0 has requirement                   │           │
│                 │ dash-bootstrap-components<=1.7.1;              │           │
│                 │ python_version > "3.0", but you have           │           │
│                 │ dash-bootstrap-components 2.0.3.               │           │
│                 │ dtale 3.17.0 has requirement dash_daq<=0.5.0,  │           │
│                 │ but you have dash-daq 0.6.0.                   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.12 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [4]:
def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [5]:
# Makes xarray dataframe (I think not a dataset) from list of s3 uris.
# This came from Solomon Negusse and I haven't really changed it.
# He said that an online forum suggested using xr.openmfdataset to open non-overlapping geotifs.
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze()
    # ).squeeze().persists()  # Need this if reading from geotifs directly, rather then creating zarrs

    return xarray_chunks

In [6]:
# Lists uris in an s3 folder, for creating zarr of them
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

# Extracts file pattern from uri. Assumes that file pattern includes _ha_yr (as it does from the LULUCF model).
def parse_pattern_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_ha_yr_\d{4}.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

# Creates a Pandas dataframe with the state_nodes codes and meanings from an Excel spreadsheet
def create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet_name):

    try:
        # Tries fetching the file from the S3 URL
        # print(f"Attempting to download file from URL: {spreadsheet}")
        response = requests.get(state_node_lookup_table_s3, timeout=10)
        response.raise_for_status()
        state_node_df = pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"Failed to download file from S3. Falling back to local file. Error: {e}")

        print(f"Reading file from local path: {state_node_lookup_table_local}")
        state_node_df = pd.read_excel(state_node_lookup_table_local, sheet_name=sheet_name)

    return state_node_df

In [7]:
def remove_FillValue(zarr_path):

    fs = fsspec.filesystem("s3", anon=False)
    mapper = fs.get_mapper(zarr_path)
    z = zarr.open_group(mapper, mode="r+")
    
    # Loop through each array and remove _FillValue if present
    for key in z.array_keys():
        arr = z[key]
        if "_FillValue" in arr.attrs:
            print(f"   Removing _FillValue from {key}")
            del arr.attrs["_FillValue"]

    print(f"   FillValues removed from {zarr_path}")

In [8]:
# Crops one input to the other input's extent.
# ref is the reference dataset that is being cropped to. 
# From long chat in https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/684749fe-7b30-800a-ba8b-c502377f2c3a
def safe_crop(ds, ref):
    return ds.sel(x=ref.x, y=ref.y, method="nearest")

# Fix floating-point precision issues
def round_coords(ds, decimals=5):
    ds = ds.assign_coords({
        'x': np.round(ds.coords['x'].values, decimals),
        'y': np.round(ds.coords['y'].values, decimals)
    })
    return ds

In [9]:
# Converts results of flox to coordinate dictionary.
# This code came from Solomon Negusse and I haven't changed it in any substantial way.
def convert_to_coord_dict(flux_results):

    print(f"   Postprocessing: {timestr()}")
    sparse_data = flux_results.data
    
    dim_names = flux_results.dims
    indices = sparse_data.coords  # tuple of arrays with indices into each dim
    values = sparse_data.data     # non-zero values
    
    coord_dict = {
        dim: flux_results.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    return coord_dict

In [10]:
# Converts flox output to dataframe and does some processing of it:
# replaces the numeric flux type with the name
# classifies specific flux types to larger groupings
# adds the interval end year to the dataframe
# adds the state node meaning to the dataframe
# converts area from m^2 to ha
def create_df(coord_dict, state_node_df):

    df = pd.DataFrame(coord_dict)
    # print(df)

    # Contextual layers to use to merge pixel_area against other analysis layers (to calculate flux/ha)
    merge_keys = ['adm0', 'land_state_node', 'year', 'primary_forest_IFL']

    # Split df into pixel area and other analysis layers
    df_area = (
        df[df['analysis_layer'] == 'pixel_area_ha']
          .rename(columns={'value': 'pixel_area_ha'})
          [merge_keys + ['pixel_area_ha']]
    )

    # Non-pixel area analysis layers
    df_other = df[df['analysis_layer'] != 'pixel_area_ha']

    # Merge area values into flux rows
    df_with_areas = df_other.merge(df_area, on=merge_keys, how='left')

    # Adds the state_node meaning and classifications to the dataframe
    df_with_areas = df_with_areas.merge(state_node_df[['state_nodes', 'meaning', 'broad_class', 'detailed_class']],
              left_on='land_state_node', right_on='state_nodes',
              how='left')
    # print("merged:", df_with_areas)

    # Replaces the year index with the actual reporting year
    df_with_areas['year'] = df_with_areas['year'] + 2016

    # Deletes redundant state node column
    df_with_areas = df_with_areas.drop(columns=['state_nodes'])

    # Converts numeric codes to ISO codes 
    # From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
    df_with_areas['adm0'] = df_with_areas.adm0.apply(lambda x: numeric_to_alpha3[x])

    df_with_areas['flux_Mg_ha'] = df_with_areas['value'] / df_with_areas['pixel_area_ha'].replace(0, pd.NA)

    return df_with_areas

In [11]:
### Value options for contextual layer values.
### Every contextual layer needs to have all possible values listed here.

# GADM v4.1 adm0 IDs (from Solomon Negusse's notebook)
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

# Primary forest value options
primary_forest_IFL_codes = np.array([0, 1], dtype=np.uint8)

# Converts numeric ISO values to ISO codes
# From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
numeric_to_alpha3 = {
    4: 'AFG', 248: 'ALA', 8: 'ALB', 12: 'DZA', 16: 'ASM', 20: 'AND', 24: 'AGO', 660: 'AIA',
    10: 'ATA', 28: 'ATG', 32: 'ARG', 51: 'ARM', 533: 'ABW', 36: 'AUS', 40: 'AUT', 31: 'AZE',
    44: 'BHS', 48: 'BHR', 50: 'BGD', 52: 'BRB', 112: 'BLR', 56: 'BEL', 84: 'BLZ', 204: 'BEN',
    60: 'BMU', 64: 'BTN', 68: 'BOL', 535: 'BES', 70: 'BIH', 72: 'BWA', 74: 'BVT', 76: 'BRA',
    86: 'IOT', 96: 'BRN', 100: 'BGR', 854: 'BFA', 108: 'BDI', 132: 'CPV', 116: 'KHM', 120: 'CMR',
    124: 'CAN', 136: 'CYM', 140: 'CAF', 148: 'TCD', 152: 'CHL', 156: 'CHN', 162: 'CXR', 166: 'CCK',
    170: 'COL', 174: 'COM', 178: 'COG', 180: 'COD', 184: 'COK', 188: 'CRI', 384: 'CIV', 191: 'HRV',
    192: 'CUB', 531: 'CUW', 196: 'CYP', 203: 'CZE', 208: 'DNK', 262: 'DJI', 212: 'DMA', 214: 'DOM',
    218: 'ECU', 818: 'EGY', 222: 'SLV', 226: 'GNQ', 232: 'ERI', 233: 'EST', 748: 'SWZ', 231: 'ETH',
    238: 'FLK', 234: 'FRO', 242: 'FJI', 246: 'FIN', 250: 'FRA', 254: 'GUF', 258: 'PYF', 260: 'ATF',
    266: 'GAB', 270: 'GMB', 268: 'GEO', 276: 'DEU', 288: 'GHA', 292: 'GIB', 300: 'GRC', 304: 'GRL',
    308: 'GRD', 312: 'GLP', 316: 'GUM', 320: 'GTM', 831: 'GGY', 324: 'GIN', 624: 'GNB', 328: 'GUY',
    332: 'HTI', 334: 'HMD', 336: 'VAT', 340: 'HND', 344: 'HKG', 348: 'HUN', 352: 'ISL', 356: 'IND',
    360: 'IDN', 364: 'IRN', 368: 'IRQ', 372: 'IRL', 833: 'IMN', 376: 'ISR', 380: 'ITA', 388: 'JAM',
    392: 'JPN', 832: 'JEY', 400: 'JOR', 398: 'KAZ', 404: 'KEN', 296: 'KIR', 408: 'PRK', 410: 'KOR',
    414: 'KWT', 417: 'KGZ', 418: 'LAO', 428: 'LVA', 422: 'LBN', 426: 'LSO', 430: 'LBR', 434: 'LBY',
    438: 'LIE', 440: 'LTU', 442: 'LUX', 446: 'MAC', 450: 'MDG', 454: 'MWI', 458: 'MYS', 462: 'MDV',
    466: 'MLI', 470: 'MLT', 584: 'MHL', 474: 'MTQ', 478: 'MRT', 480: 'MUS', 175: 'MYT', 484: 'MEX',
    583: 'FSM', 498: 'MDA', 492: 'MCO', 496: 'MNG', 499: 'MNE', 500: 'MSR', 504: 'MAR', 508: 'MOZ',
    104: 'MMR', 516: 'NAM', 520: 'NRU', 524: 'NPL', 528: 'NLD', 540: 'NCL', 554: 'NZL', 558: 'NIC',
    562: 'NER', 566: 'NGA', 570: 'NIU', 574: 'NFK', 807: 'MKD', 580: 'MNP', 578: 'NOR', 512: 'OMN',
    586: 'PAK', 585: 'PLW', 275: 'PSE', 591: 'PAN', 598: 'PNG', 600: 'PRY', 604: 'PER', 608: 'PHL',
    612: 'PCN', 616: 'POL', 620: 'PRT', 630: 'PRI', 634: 'QAT', 638: 'REU', 642: 'ROU', 643: 'RUS',
    646: 'RWA', 652: 'BLM', 654: 'SHN', 659: 'KNA', 662: 'LCA', 663: 'MAF', 666: 'SPM', 670: 'VCT',
    882: 'WSM', 674: 'SMR', 678: 'STP', 682: 'SAU', 686: 'SEN', 688: 'SRB', 690: 'SYC', 694: 'SLE',
    702: 'SGP', 534: 'SXM', 703: 'SVK', 705: 'SVN', 90: 'SLB', 706: 'SOM', 710: 'ZAF', 239: 'SGS',
    728: 'SSD', 724: 'ESP', 144: 'LKA', 729: 'SDN', 740: 'SUR', 744: 'SJM', 752: 'SWE', 756: 'CHE',
    760: 'SYR', 158: 'TWN', 762: 'TJK', 834: 'TZA', 764: 'THA', 626: 'TLS', 768: 'TGO', 772: 'TKL',
    776: 'TON', 780: 'TTO', 788: 'TUN', 792: 'TUR', 795: 'TKM', 796: 'TCA', 798: 'TUV', 800: 'UGA',
    804: 'UKR', 784: 'ARE', 826: 'GBR', 840: 'USA', 581: 'UMI', 858: 'URY', 860: 'UZB', 548: 'VUT',
    862: 'VEN', 704: 'VNM', 92: 'VGB', 850: 'VIR', 876: 'WLF', 732: 'ESH', 887: 'YEM', 894: 'ZMB',
    716: 'ZWE', 0: 'NA'
}

Code to run zonal stats

In [12]:
# General zonal stat run properties

model_version = "version_1_0_3_AUS_only_chunk_1x4000x4000"  # model version, from s3 paths that are being read
run_date = "20251209"   # model run date, from s3 paths that are being read
chunk_size = 4000  # pixels

interval_label = '2016'

# s3 folders for model outputs being analyzed
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_vegetation/{model_version}/"  # Model output path, for inputs to zonal stats

# Analysis layer s3 paths
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_non_CO2_folder = f"{output_path}gross_emissions__all_C_pools__non_CO2_only__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_all_gases_folder = f"{output_path}net_flux__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/"

# Rechunked mega-zarr
model_mega_zarr_s3_path = f"{output_path}mega_zarr/standard_model/annual_intervals/{chunk_size}_pixels/{run_date}/"

# zarrs for layers not from the flux model (only need to created once)
# They are in a central folder, not with their specific geotif tile sets (at least for now-- we could change this)
adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
adm0_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/GADM4_1_adm0_global/20251209_fillValue_removed/global_GADM41_adm0_20251209.zarr"

pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"
pixel_area_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/pixel_area/20251209_fillValue_removed/global_pixel_area_20251209.zarr"

primary_forest_IFL_folder = "s3://gfw2-data/climate/carbon_model/ifl_primary_merged/processed/20200724/"
primary_forest_IFL_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/IFL2000_tropical_primary_forest_2001/20251209_fillValue_removed/ifl_primary_forest_merged_20251209.zarr"

# Spreadsheet for state_node meanings (local computer and s3 locations)
state_node_lookup_table_local = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx"
state_node_lookup_table_s3 = "http://gfw2-data.s3.amazonaws.com/climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx"
sheet = "v102_20251027"

In [13]:
# # %%time

# # # CREATES ZARRS FOR INPUTS NOT GENERATED BY THE AFOLU MODEL
# # # THIS SHOULD ONLY EVER HAVE TO BE DONE ONCE FOR EACH INPUT
# # # Creating adm0, pixel area and IFL/primary forest used 54 credits $3.05 AWS charges, and 7 minutes (50 r7g.2xlarge workers).
# # # https://cloud.coiled.io/clusters/1311444/account/wri-forest-research/information?workspace=WRI-forest-research

# # print(f"Reading inputs that apply to all intervals: {timestr()}")

# # GADM adm0
# adm0_uris = list_folder_uris(adm0_folder)
# print("adm0_folder:", adm0_folder)
# print(adm0_uris[0])
# print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")

# print(f"   Reading adm0: {timestr()}")
# adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
# adm0_xarray_chunks['band_data'] = adm0_xarray_chunks['band_data'].astype('uint16')  # adm0 should be uint16 but make_xarray_chunks makes it float64 for some reason
# # print("adm0_xarray_chunks:", adm0_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring adm0: {timestr()}")
# adm0_xarray_chunks.to_zarr(adm0_zarr_path, mode='w')
# remove_FillValue(adm0_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring adm0: {timestr()}")


# # Pixel area
# pixel_area_uris = list_folder_uris(pixel_area_folder)
# print("pixel_area_folder:", pixel_area_folder)
# print(pixel_area_uris[0])
# print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

# print(f"   Reading pixel_area: {timestr()}")
# pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)
# print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

# print(f"   zarring pixel_area: {timestr()}")
# pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_path, mode='w')
# remove_FillValue(pixel_area_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring pixel_area: {timestr()}")


# # Humid tropical primary forest/IFL merged
# primary_forest_IFL_uris = list_folder_uris(primary_forest_IFL_folder)
# print("primary_forest_IFL_folder:", primary_forest_IFL_folder)
# print(primary_forest_IFL_uris[0])
# print(f"Tile count in {primary_forest_IFL_folder}: {len(primary_forest_IFL_uris)}")

# print(f"   Reading primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks = make_xarray_chunks(primary_forest_IFL_uris, chunk_size)
# primary_forest_IFL_xarray_chunks['band_data'] = primary_forest_IFL_xarray_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("primary_forest_IFL_xarray_chunks:", primary_forest_IFL_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks.to_zarr(primary_forest_IFL_zarr_path, mode='w')
# remove_FillValue(primary_forest_IFL_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring primary_forest_IFL: {timestr()}")

In [14]:
# # Get filename patterns from the GeoTIFF URIs if you rely on them later
# gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval_label)
# gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)
# gross_emis_non_CO2_folder_interval = gross_emis_non_CO2_folder.replace("INTERVAL", interval_label)
# gross_emis_non_CO2_uris = list_folder_uris(gross_emis_non_CO2_folder_interval)
# gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval_label)
# gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
# gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval_label)
# gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
# net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval_label)
# net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
# net_flux_all_pools_all_gases_folder_interval = net_flux_all_pools_all_gases_folder.replace("INTERVAL", interval_label)
# net_flux_all_pools_all_gases_uris = list_folder_uris(net_flux_all_pools_all_gases_folder_interval)
# node_folder_interval = node_folder.replace("INTERVAL", interval_label)
# node_tile_year_uris = list_folder_uris(node_folder_interval)
# 
# gross_emis_CO2_output_pattern       = parse_pattern_from_uri(gross_emis_CO2_uris)
# gross_emis_non_CO2_output_pattern   = parse_pattern_from_uri(gross_emis_non_CO2_uris)
# gross_emis_all_gases_output_pattern = parse_pattern_from_uri(gross_emis_all_gases_uris)
# gross_remv_all_pools_output_pattern = parse_pattern_from_uri(gross_remv_all_pools_uris)
# net_flux_CO2_output_pattern         = parse_pattern_from_uri(net_flux_all_pools_CO2_uris)
# net_flux_all_gases_output_pattern   = parse_pattern_from_uri(net_flux_all_pools_all_gases_uris)
# node_output_pattern                 = 'land_state_node'

gross_emis_CO2_output_pattern       = "gross_emissions__all_C_pools__CO2_only__MgCO2"
gross_emis_non_CO2_output_pattern   = "gross_emissions__all_C_pools__non_CO2_only__MgCO2e"
gross_emis_all_gases_output_pattern = "gross_emissions__all_C_pools__all_gases__MgCO2e"
gross_remv_all_pools_output_pattern = "gross_removals__all_C_pools__MgCO2"
net_flux_CO2_output_pattern         = "net_flux__all_C_pools__CO2_only__MgCO2"
net_flux_all_gases_output_pattern   = "net_flux__all_C_pools__all_gases__MgCO2e"
node_output_pattern                 = "land_state_node"

print(gross_emis_CO2_output_pattern)
print(gross_emis_non_CO2_output_pattern)
print(gross_emis_all_gases_output_pattern)
print(gross_remv_all_pools_output_pattern)
print(net_flux_CO2_output_pattern)
print(gross_emis_CO2_output_pattern)
print(net_flux_all_gases_output_pattern)
print(node_output_pattern)

gross_emissions__all_C_pools__CO2_only__MgCO2
gross_emissions__all_C_pools__non_CO2_only__MgCO2e
gross_emissions__all_C_pools__all_gases__MgCO2e
gross_removals__all_C_pools__MgCO2
net_flux__all_C_pools__CO2_only__MgCO2
gross_emissions__all_C_pools__CO2_only__MgCO2
net_flux__all_C_pools__all_gases__MgCO2e
land_state_node


In [15]:
# Opens flux model output mega-zarr
ds_all_global = xr.open_zarr(model_mega_zarr_s3_path, consolidated=False)
ds_all_global

<xarray.Dataset> Size: 998TB
Dimensions:                                             (year: 9, y: 720000,
                                                         x: 1440000)
Coordinates:
  * y                                                   (y) float64 6MB 90.0 ...
  * year                                                (year) int64 72B 0 ... 8
  * x                                                   (x) float64 12MB -180...
Data variables: (12/29)
    carbon_density__deadwood_C__MgC                     (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    carbon_density__AGC__MgC                            (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    carbon_density__litter_C__MgC                       (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    carbon_density__BGC__MgC                            (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    gross_emissions__all_C_pools__CO2_only__MgCO2       (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    gross_emissions__all_C_pools__all_gases__MgCO2e     (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    ...                                                  ...
    net_flux__all_C_pools__CO2_only__MgCO2              (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    net_flux__BGC__MgCO2                                (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    net_flux__deadwood_C__MgCO2                         (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    spatial_ref                                         int32 4B ...
    net_flux__litter_C__MgCO2                           (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>
    net_flux__all_C_pools__all_gases__MgCO2e            (year, y, x) float32 37TB dask.array<chunksize=(1, 4000, 4000), meta=np.ndarray>

In [16]:
ds_all_global.chunksizes

Frozen({'year': (1, 1, 1, 1, 1, 1, 1, 1, 1), 'y': (4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4

In [17]:
# Opens non-model output zarrs
adm0_xr = xr.open_zarr(adm0_zarr_path, consolidated=False).rename_vars(band_data='adm0')
pixel_area_xr = xr.open_zarr(pixel_area_zarr_path, consolidated=False).rename_vars(band_data='pixel_area')
primary_forest_IFL_xr = xr.open_zarr(primary_forest_IFL_zarr_path, consolidated=False).rename_vars(band_data='primary_forest_IFL')
pixel_area_xr
# print(primary_forest_IFL_xr)

<xarray.Dataset> Size: 6TB
Dimensions:      (y: 560000, x: 1440000)
Coordinates:
  * y            (y) float64 4MB 80.0 80.0 80.0 80.0 ... -60.0 -60.0 -60.0 -60.0
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
    band         int64 8B ...
Data variables:
    pixel_area   (y, x) float64 6TB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    spatial_ref  int64 8B ...

In [18]:
# Creates dataframe of state_node codes and meanings
state_node_df = create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet)
node_codes = np.array(list(state_node_df['state_nodes']), dtype=np.uint32)
# node_codes

In [19]:
%%time

#TODO Solomon says that analysis layers need to be recast to float64: "We found out that aggregating all the pixel values over large areas hits float32 limits which will give very wrong results."
selected_vars = ["gross_emissions__all_C_pools__CO2_only__MgCO2",
                 "gross_emissions__all_C_pools__non_CO2_only__MgCO2e",
                 "gross_emissions__all_C_pools__all_gases__MgCO2e",
                 "gross_removals__all_C_pools__MgCO2",
                 "net_flux__all_C_pools__CO2_only__MgCO2",
                 "net_flux__all_C_pools__all_gases__MgCO2e"
                ]

# Define bounding box
# west, south, east, north = 124, -30, 125, -29   # Test chunk
west, south, east, north = 112, -56, 160, -9    # Australia bounding box (48x47 deg)

ds_all_global_selected_vars = ds_all_global[selected_vars]
ds_all_global_selected_vars

# Need to get land_state_node dataset from the 9x4000x4000 zarr because I forgot to rechunk it with the output fluxes
model_mega_zarr_s3_path_land_state_node = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_vegetation/{model_version}/mega_zarr/standard_model/annual_intervals/4000_pixels/{run_date}/"
ds_all_global_land_state_node = xr.open_zarr(model_mega_zarr_s3_path_land_state_node, consolidated=False)

print(f"Rounding coordinates: {timestr()}")
reference = round_coords(pixel_area_xr["pixel_area"])
ds_all_global_selected_vars = round_coords(ds_all_global_selected_vars)
adm0_xr = round_coords(adm0_xr)
primary_forest_IFL_xr = round_coords(primary_forest_IFL_xr)
land_state_node = round_coords(ds_all_global_land_state_node["land_state_node"])

print(f"Cropping: {timestr()}")
pixel_area_aligned                       = reference
adm0_aligned                             = safe_crop(adm0_xr, reference)
primary_forest_IFL_aligned               = safe_crop(primary_forest_IFL_xr, reference)
land_state_node_aligned                  = safe_crop(land_state_node, reference)
ds_all_global_selected_vars_aligned      = safe_crop(ds_all_global_selected_vars, reference)

print(f"Creating flux cube: {timestr()}")
# List of selected variable names (already aligned and cropped)
selected_datasets = list(ds_all_global_selected_vars_aligned.data_vars)

# Expand pixel_area to match shape of flux variables
pixel_area_expanded = pixel_area_aligned.expand_dims(year=ds_all_global_selected_vars_aligned.year)

# Use the exact same x/y coordinates for both
x_coords = reference.coords['x']
y_coords = reference.coords['y']

# Replace coords in both sources
# pixel_area_expanded = pixel_area_expanded.assign_coords(x=x_coords, y=y_coords)
ds_all_global_selected_vars_aligned = ds_all_global_selected_vars_aligned.assign_coords(x=x_coords, y=y_coords)

# Multiply each flux var by pixel_area
flux_layers = []
for var in selected_datasets:
    flux_scaled = ((ds_all_global_selected_vars_aligned[var] * pixel_area_expanded) / 10000).astype("float32")
    flux_layers.append(flux_scaled)


# Converts pixel_area from m² to hectares, then adds to the list of layers to analyze
pixel_area_layer = (pixel_area_expanded / 10000).astype("float32")
flux_layers.append(pixel_area_layer)

# Also updates the list of analysis layer names
selected_datasets.append("pixel_area_ha")

# Stack into one flux cube: shape (analysis_layer, year, y, x)
flux_cube = xr.concat(flux_layers, dim="analysis_layer")

# Set the analysis_layer coordinate names
flux_cube = flux_cube.assign_coords(
    analysis_layer=("analysis_layer", selected_datasets)
)
flux_cube = round_coords(flux_cube)

# Subset the flux cube by x/y coordinates
flux_cube_subset = flux_cube.sel(
    x=slice(west, east),
    y=slice(north, south)  # Note: y typically decreases from top to bottom
)
adm0_aligned_subset = adm0_aligned.sel(x=slice(west, east), y=slice(north, south))
primary_forest_IFL_aligned_subset = primary_forest_IFL_aligned.sel(x=slice(west, east), y=slice(north, south))
land_state_node_aligned_subset = land_state_node_aligned.sel(x=slice(west, east), y=slice(north, south))
pixel_area_expanded_subset = pixel_area_expanded.sel(x=slice(west, east), y=slice(north, south))

print("Flux cube x range:", flux_cube_subset.coords['x'].values.min(), flux_cube_subset.coords['x'].values.max(), "len:", len(flux_cube_subset.coords['x']))
print("Pixel area x range:", pixel_area_expanded_subset.coords['x'].values.min(), pixel_area_expanded_subset.coords['x'].values.max(), "len:", len(pixel_area_expanded_subset.coords['x']))
print("ADM0 x range:", adm0_aligned_subset["adm0"].coords['x'].values.min(), adm0_aligned_subset["adm0"].coords['x'].values.max(), "len:", len(adm0_aligned_subset["adm0"].coords['x']))
print("land_state_node x range:", land_state_node_aligned_subset.coords['x'].values.min(), land_state_node_aligned_subset.coords['x'].values.max(), "len:", len(land_state_node_aligned_subset.coords['x']))
print("IFL x range:", primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x'].values.min(), primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x'].values.max(), "len:", len(primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x']))

# Final alignment 
print(f"Aligning: {timestr()}")
flux_cube_subset, pixel_area_expanded_subset, adm0_aligned_subset, primary_forest_IFL_aligned_subset, land_state_node_aligned_subset = xr.align(
    flux_cube_subset, pixel_area_expanded_subset, adm0_aligned_subset["adm0"], primary_forest_IFL_aligned_subset["primary_forest_IFL"], land_state_node_aligned_subset, join="override"
)

flux_cube_subset = flux_cube_subset.persist()

print(f"Computing: {timestr()}")
results = xarray_reduce(
    flux_cube_subset,
    *(adm0_aligned_subset, land_state_node_aligned_subset, primary_forest_IFL_aligned_subset, flux_cube_subset["year"]),
    func='sum',
    expected_groups=(gadm_adm0_ids, node_codes, primary_forest_IFL_codes, list(range(9))),
    group_dims=["year"],
    reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),
    fill_value=0
).compute()
coord_dict = convert_to_coord_dict(results)
df = create_df(coord_dict, state_node_df)
df.head()

Rounding coordinates: 20251219_10_10_02
Cropping: 20251219_10_10_02
Creating flux cube: 20251219_10_11_04
Flux cube x range: 112.00012 159.99988 len: 192000
Pixel area x range: 112.00012 159.99988 len: 192000
ADM0 x range: 112.00012 159.99988 len: 192000
land_state_node x range: 112.00012 159.99987 len: 192000
IFL x range: 112.00012 159.99988 len: 192000
Aligning: 20251219_10_13_40


/home/dagibbs22/miniforge3/envs/coiled_20251119/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 285.24 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Computing: 20251219_10_15_20


/home/dagibbs22/miniforge3/envs/coiled_20251119/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 125.53 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


FutureCancelledError: ('reshape-c37e0965e1335e7eedd35b4b6c6d9e97', 6, 0, 0, 0, 0) cancelled for reason: scheduler-connection-lost.
Client lost the connection to the scheduler. Please check your connection and re-run your work.

In [20]:
df = create_df(coord_dict, state_node_df)
df

NameError: name 'coord_dict' is not defined

In [142]:
print(df[(df.analysis_layer == 'gross_emissions__all_C_pools__CO2_only__MgCO2') & (df.year == 2016)]['value'].sum().round())
print(df[(df.analysis_layer == 'gross_emissions__all_C_pools__CO2_only__MgCO2') & (df.year == 2017)]['value'].sum().round())
print(df[(df.analysis_layer == 'net_flux__all_C_pools__all_gases__MgCO2e') & (df.year == 2016)]['value'].sum().round())
print(df[(df.analysis_layer == 'net_flux__all_C_pools__all_gases__MgCO2e') & (df.year == 2017)]['value'].sum().round())

115818776.0
91429310.0
-309706980.0
-349966140.0


In [143]:
# Export to a csv so data can be used in Excel or reused
df.to_csv('/mnt/c/GIS/vegetation_flux_zonal_stats_AUS_v_1_0_3__20251211.csv', index=False)

In [103]:
df_wide = df.pivot(index=['land_state_node', 'meaning', 'broad_class', 'detailed_class', 'year', 'primary_forest_IFL', 'adm0'], columns="analysis_layer", values="value").reset_index()
# df_wide = df.pivot(index=['land_state_node', 'year', 'primary_forest_IFL', 'adm0'], columns="analysis_layer", values="value").reset_index()
df_wide

analysis_layer,land_state_node,meaning,broad_class,detailed_class,year,primary_forest_IFL,adm0,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_removals__all_C_pools__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,pixel_area_ha
0,10000000,Before mangrove gain (mangrove mask),tree,mangrove_other,2016,0,AUS,NaN,NaN,NaN,NaN,NaN,NaN,8.084027e+03
1,10000000,Before mangrove gain (mangrove mask),tree,mangrove_other,2016,0,NA,NaN,NaN,NaN,NaN,NaN,NaN,2.104000e+03
2,10000000,Before mangrove gain (mangrove mask),tree,mangrove_other,2016,0,PNG,NaN,NaN,NaN,NaN,NaN,NaN,2.560480e+01
3,10000000,Before mangrove gain (mangrove mask),tree,mangrove_other,2016,1,AUS,NaN,NaN,NaN,NaN,NaN,NaN,1.199906e+01
4,10000000,Before mangrove gain (mangrove mask),tree,mangrove_other,2016,1,NA,NaN,NaN,NaN,NaN,NaN,NaN,5.313004e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2036,70000000,Not in decision tree,no_flux,no_flux,2024,0,NA,NaN,NaN,NaN,NaN,NaN,NaN,1.672736e+08
2037,70000000,Not in decision tree,no_flux,no_flux,2024,0,PNG,NaN,NaN,NaN,NaN,NaN,NaN,1.008444e+04
2038,70000000,Not in decision tree,no_flux,no_flux,2024,1,AUS,NaN,NaN,NaN,NaN,NaN,NaN,7.747959e+04
2039,70000000,Not in decision tree,no_flux,no_flux,2024,1,NA,NaN,NaN,NaN,NaN,NaN,NaN,1.683903e+03


In [104]:
walker = pyg.walk(df_wide)

Box(children=(HTML(value='<div id="ifr-pyg-2" style="height: auto">\n    <head>\n        <meta http-equiv="Con…

In [49]:
try:
    client.shutdown()
except Exception:
    pass